<a href="https://colab.research.google.com/github/JoBa2211/-reproductor-cortador/blob/main/Jose_Guadalupe_Practica1_Conceptos_SO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💻 Sistemas Operativos - Exploración de la Estructura del SO y Recursos del Sistema
**Asignatura:** Fundamentos y Diseño de Sistemas Operativos (6.º Semestre)    
**Unidad I:** Conceptos y Evolución de los Sistemas Operativos  
**Tiempo estimado:** 2 horas (Asíncrono)  
**Repositorio Base:** Integración con GitHub  

---

## 1. Objetivos de Aprendizaje
1. Identificar la estructura física y lógica de la máquina virtual Linux subyacente en Google Colab.
2. Analizar el rol del Kernel en la administración de recursos hardware (CPU, Memoria, E/S).
3. Ejecutar y comprender llamadas al sistema (Syscalls) y la arquitectura en modo Kernel/Usuario.

---


## 2. Conexión Teórica (NotebookLM)
**Antes de continuar:** Accede a tu entorno de **NotebookLM del Curso**, abre la sección de la *Unidad I* y consulta:
---

- *¿Qué es el Kernel de Linux y cómo abstrae el hardware?*  
**RESPUESTA:**
El kernel de Linux es la parte central y más fundamental del sistema operativo, diseñada para ejecutarse en modo núcleo (o modo supervisor). En este modo de operación, el kernel tiene acceso completo a todo el hardware de la máquina y puede ejecutar cualquier instrucción que el procesador sea capaz de procesar. Reside de manera permanente en la memoria principal y, a diferencia de los sistemas basados en un micronúcleo, posee una estructura monolítica. Esto significa que toda su funcionalidad esencial (como la gestión de procesos, la memoria, las redes y los controladores de dispositivos) se ejecuta como un único programa binario extenso que comparte el mismo espacio de direcciones en modo protegido. A pesar de este diseño clásico, Linux logra una gran flexibilidad y dinamismo mediante el mecanismo de módulos cargables, los cuales permiten añadir o reemplazar controladores y funciones del núcleo en tiempo real sin necesidad de reiniciar la computadora.

Internamente, el kernel de Linux no realiza una distinción rígida entre procesos e hilos; en su lugar, representa cualquier contexto de ejecución como una tarea (`task`) mediante la estructura de datos `task_struct`.

**¿Cómo abstrae el hardware?**
La tarea primordial del kernel es ocultar la interfaz compleja, desordenada e inconsistente del hardware físico y presentar a las aplicaciones una máquina virtual extendida con abstracciones elegantes, simples y consistentes a través de tres pilares fundamentales:

* **Abstracción de Entrada/Salida (E/S) y Almacenamiento:** Linux unifica el acceso a todos los dispositivos físicos tratándolos como si fueran archivos asociados a un nombre de ruta en el directorio `/dev` (por ejemplo, `/dev/sda` para discos).
* **Sistema de Archivos Virtual (VFS):** Define un modelo común de archivos y operaciones. Gracias a esto, una llamada al sistema estándar como `read` o `write` funciona de manera idéntica ya sea que la aplicación esté interactuando con un archivo regular en el disco, una tubería de comunicación, un socket de red o un dispositivo de hardware.
* **Controladores de dispositivos (Drivers):** Actúan como traductores que reciben peticiones abstractas del software y las convierten en secuencias concretas de escritura en los registros de control de los chips físicos del hardware.


* **Abstracción de la Memoria Principal:** Para evitar que las aplicaciones compitan directamente por la memoria física o dañen el espacio de direcciones de otros procesos, Linux implementa el esquema de memoria virtual.
* **Paginación bajo demanda:** El kernel divide el espacio lógico de cada proceso en páginas y las proyecta sobre marcos de memoria física mediante la Unidad de Gestión de Memoria (MMU) del procesador y tablas de páginas multinivel.
* **Asignación de recursos:** Administra los marcos físicos usando el algoritmo *buddy* (sistema de colegas) para bloques grandes y el asignador de losas (*slab allocator*) para porciones y cachés de memoria más pequeñas destinadas a objetos internos del sistema.


* **Abstracción del Procesador (CPU):** El procesador físico se multiplexa y se transforma conceptualmente en múltiples CPU virtuales independientes.
* **Planificación y cambio de contexto:** El procesador se asigna a las distintas tareas activas por intervalos de tiempo (*quantums*). Al cambiar de proceso, el kernel salva el estado de los registros de hardware en el descriptor `task_struct` de la tarea actual y restaura el de la siguiente.
* **Protección y llamadas al sistema:** La CPU se mantiene en modo usuario sin privilegios. Cuando una aplicación requiere acceder al hardware, realiza una llamada al sistema provocando una instrucción de trampa de software (`TRAP`), lo que conmuta de forma segura el procesador a modo núcleo protegido para ejecutar la operación de manera controlada.

- *¿Qué función cumple la jerarquía de memoria en un SO moderno?*  
**RESPUESTA:**
La jerarquía de memoria en un sistema operativo moderno cumple la función crítica de resolver la discrepancia existente entre la velocidad del procesador, el elevado costo de las tecnologías de almacenamiento rápido y las masivas necesidades de capacidad de los programas actuales.

Dado que los programadores desean idealmente una memoria privada, infinitamente grande, infinitamente rápida, no volátil y económica —características que la tecnología actual no puede proveer en un único componente físico—, la jerarquía combina múltiples niveles de tecnologías complementarias.

A medida que se desciende por los niveles de la jerarquía (desde los registros internos del procesador, las cachés L1/L2/L3, la memoria principal DRAM, hasta el almacenamiento secundario como discos SSD/HDD y el almacenamiento terciario), disminuye el costo por bit, aumenta significativamente la capacidad de almacenamiento y se incrementa el tiempo de acceso de manera exponencial.

El sistema operativo, asistido por el hardware de la máquina, gestiona esta jerarquía a través de las siguientes funciones clave:

* **Abstracción y gestión eficiente (El Administrador de Memoria):** El administrador de memoria del sistema operativo se encarga de abstraer la complejidad física de esta jerarquía para presentar a las aplicaciones un modelo de memoria lógico útil y transparente. Sus funciones directas consisten en:
* Llevar el registro de qué zonas de la memoria están libres y cuáles ocupadas.
* Asignar memoria a los procesos de forma dinámica cuando la solicitan y desasignarla cuando terminan.
* Controlar la migración de información entre niveles adyacentes de la jerarquía (por ejemplo, subir bloques del disco a la memoria principal o viceversa).


* **Optimización del tiempo de acceso mediante la proximidad referencial:** La jerarquía de memoria es viable y rentable gracias al principio de proximidad de referencias (localidad referencial), el cual demuestra que los programas en ejecución tienden a agrupar sus accesos a memoria en zonas localizadas durante intervalos cortos de tiempo. Esta proximidad se presenta en dos dimensiones:
* **Proximidad temporal:** Si un dato o instrucción se referencia en un momento dado, existe una alta probabilidad de que sea accedido nuevamente en un futuro próximo.
* **Proximidad espacial:** Si se accede a una posición de memoria, es muy probable que se referencien posiciones contiguas a ella en breve (lo que justifica transferir datos en bloques, como páginas o líneas de caché).
* El objetivo primordial de la gestión de la jerarquía es conseguir que residan en las memorias más rápidas aquellas porciones del programa que están siendo activamente referenciadas, optimizando el tiempo medio de acceso efectivo del sistema y manteniendo una tasa de aciertos alta.


* **Implementación y soporte de la memoria virtual:** La interacción automatizada entre la memoria principal (memoria real) y el disco duro (dispositivo de paginación o *swap*) permite la existencia de la memoria virtual, cumpliendo funciones trascendentales:
* **Ejecución de programas grandes:** Mantiene en la memoria física solo aquellas porciones necesarias para la ejecución inmediata (conjunto residente), dejando el resto en el disco de respaldo.
* **Maximización de la multiprogramación:** Aloja un mayor número de procesos concurrentes listos para ejecutar, elevando el rendimiento y la productividad de la CPU al aprovechar los tiempos de bloqueo por E/S de unos procesos para ejecutar otros.
* **Prevención de la hiperpaginación (*thrashing*):** El SO monitorea el comportamiento del sistema para asegurar que el conjunto residente de páginas de cada proceso contenga su conjunto de trabajo real, evitando que el procesador pase la mayor parte del tiempo atendiendo fallos de página improductivos.


* **Mantenimiento de la coherencia de datos:** Un efecto colateral de tener múltiples niveles es la existencia de varias copias de la misma información en distintas capas de almacenamiento. Al realizar una operación de escritura sobre un nivel rápido, se produce una falta de coherencia con los niveles inferiores (dejando la porción modificada en estado "sucio"). El sistema operativo, con apoyo de hardware como la MMU o controladores de acceso directo a memoria (DMA), se encarga de aplicar políticas de actualización (como la escritura retrasada o perezosa) para volcar y unificar de forma ordenada los datos modificados hacia los medios permanentes o de almacenamiento secundario.

---

## 3. Sección Práctica 1: Inspección de Recursos del Sistema (Bash / CLI)

En esta sección exploraremos el sistema sobre el cual se ejecuta este Cuaderno (Servidor Linux provisto por Google).

### [CÓDIGO 1.1] Identificación del Kernel y la Distribución Linux
```bash
# Ejecutamos comandos Bash usando el prefijo !
!echo "=== INFORMACIÓN DEL KERNEL ==="
!uname -a
!echo ""
!echo "=== DISTRIBUCIÓN DE LINUX ==="
!cat /etc/os-release | grep -E "PRETTY_NAME|VERSION"

In [ ]:
!echo "=== INFORMACIÓN DEL KERNEL ==="
!uname -a
!echo ""
!echo "=== DISTRIBUCIÓN DE LINUX ==="
!cat /etc/os-release | grep -E "PRETTY_NAME|VERSION"

=== INFORMACIÓN DEL KERNEL ===
Linux d4d866c0ff0e 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux

=== DISTRIBUCIÓN DE LINUX ===
PRETTY_NAME="Ubuntu 22.04.5 LTS"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy


### [CÓDIGO 1.2] Inspección de la CPU (Recurso Procesador)
```bash
!echo "=== ARQUITECTURA Y NÚCLEOS DE CPU ==="
!lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Architecture"

In [ ]:
!echo "=== ARQUITECTURA Y NÚCLEOS DE CPU ==="
!lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Architecture"

=== ARQUITECTURA Y NÚCLEOS DE CPU ===
Architecture:                            x86_64
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Core(s) per socket:                      1
NUMA node0 CPU(s):                       0,1


### [CÓDIGO 1.3] Inspección de la Memoria RAM y Jerarquía de Almacenamiento
```bash
!echo "=== ESTADO DE LA MEMORIA RAM (en MB) ==="
!free -m
!echo ""
!echo "=== SISTEMA DE ARCHIVOS Y DISCO DURA ==="
!df -h /

In [ ]:
!echo "=== ESTADO DE LA MEMORIA RAM (en MB) ==="
!free -m
!echo ""
!echo "=== SISTEMA DE ARCHIVOS Y DISCO DURO ==="
!df -h /

=== ESTADO DE LA MEMORIA RAM (en MB) ===
               total        used        free      shared  buff/cache   available
Mem:           12975         746        8979           2        3249       11946
Swap:              0           0           0

=== SISTEMA DE ARCHIVOS Y DISCO DURO ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  19% /


### Pregunta de Análisis 1 (Responde en la celda de texto):
Basándote en las lecturas de Tanenbaum en NotebookLM y en los resultados obtenidos arriba:

1. ¿Qué versión de Kernel Linux está ejecutando la máquina virtual de Colab?  
**RESPUESTA:  
*Linux ba983ae5e2cc 6.6.122+*

2. ¿Cuántos núcleos de procesador y cuánta memoria RAM tiene asignados tu entorno virtual?  
**RESPUESTA:  
*Nucleos de procesador 2*   
*Memoria ram asignada 12975 megas*

## 4. Sección Práctica 2: Abstracción de Hardware a través del Sistema `/proc`

En Linux, `/proc` es un **pseudo-sistema de archivos**. No ocupa espacio en disco; es una interfaz directa mediante la cual el Kernel expone información sobre el estado de los recursos y procesos del sistema.

### [CÓDIGO 2.1] Exploración del sistema /proc
```bash
!echo "=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ==="
!head -n 10 /proc/meminfo
!echo ""
!echo "=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ==="
!cat /proc/uptime

In [ ]:
!echo "=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ==="
!head -n 10 /proc/meminfo
!echo ""
!echo "=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ==="
!cat /proc/uptime

=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ===
MemTotal:       13286944 kB
MemFree:         9171932 kB
MemAvailable:   12234716 kB
Buffers:          115120 kB
Cached:          3144840 kB
SwapCached:            0 kB
Active:           464684 kB
Inactive:        3445408 kB
Active(anon):       1424 kB
Inactive(anon):   650912 kB

=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ===
421.27 646.99


### [CÓDIGO 2.2] Visualización del árbol de llamadas e interrupciones del hardware
```bash
!echo "=== INTERRUPCIONES DE HARDWARE REGISTRADAS ==="
!head -n 20 /proc/interrupts

In [ ]:
!echo "=== INTERRUPCIONES DE HARDWARE REGISTRADAS ==="
!head -n 20 /proc/interrupts

=== INTERRUPCIONES DE HARDWARE REGISTRADAS ===
           CPU0       CPU1       
  0:        257          0   IO-APIC   0-edge      timer
  1:          0          9   IO-APIC   1-edge      i8042
  4:       1547       1247   IO-APIC   4-edge      ttyS0
  8:          0          0   IO-APIC   8-edge      rtc0
  9:          0          0   IO-APIC   9-fasteoi   acpi
 10:          0       1386   IO-APIC  10-fasteoi   virtio2
 12:          3          0   IO-APIC  12-edge      i8042
 24:          0          0  PCI-MSIX-0000:00:06.0   0-edge      virtio3-config
 25:          0         35  PCI-MSIX-0000:00:06.0   1-edge      virtio3-input
 26:          0          0  PCI-MSIX-0000:00:03.0   0-edge      virtio0-config
 27:          0          0  PCI-MSIX-0000:00:03.0   1-edge      virtio0-control
 28:          0          0  PCI-MSIX-0000:00:03.0   2-edge      virtio0-event
 29:          0     122520  PCI-MSIX-0000:00:03.0   3-edge      virtio0-request
 30:          0          0  PCI-MSIX-0000:00:0

## 5. Sección Práctica 3: Primer Contacto con Bajo Nivel (C / Syscalls POSIX)

A continuación, crearemos, compilaremos y ejecutaremos un programa en C para observar directamente las **Llamadas al Sistema** (`sys_write`, `sys_getpid`) que interactúan entre el **Modo Usuario** y el **Modo Kernel**.

```bash
%%writefile syscall_demo.c
/*
 * Práctica 1 - Introducción a Llamadas al Sistema POSIX
 * Este programa realiza llamadas al sistema para obtener datos del proceso.
 */

#include <stdio.h>
#include <unistd.h>
#include <sys/types.h>

int main() {
    // getpid() es una llamada al sistema POSIX para solicitar al Kernel el ID del proceso actual
    pid_t process_id = getpid();
    pid_t parent_id = getppid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("--------------------------------------------------\n");

    return 0;
}

In [ ]:
%%writefile syscall_demo.c
/*
 * Práctica 1 - Introducción a Llamadas al Sistema POSIX
 * Este programa realiza llamadas al sistema para obtener datos del proceso.
 */

# include <stdio.h>
# include <unistd.h>
# include <sys/types.h>

int main() {
    // getpid() es una llamada al sistema POSIX para solicitar al Kernel el ID del proceso actual
    pid_t process_id = getpid();
    pid_t parent_id = getppid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("--------------------------------------------------\n");

    return 0;
}

Overwriting syscall_demo.c


### [CÓDIGO 3.2] Compilación y Ejecución del Código C usando GCC
```bash
!gcc -Wall syscall_demo.c -o syscall_demo
!./syscall_demo

In [ ]:
!gcc -Wall syscall_demo.c -o syscall_demo
!./syscall_demo

--------------------------------------------------
[MODO USUARIO] Ejecutando programa en C...
[LLAMADA AL SISTEMA] PID del Proceso Actual: 6316
[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): 4468
--------------------------------------------------


### [CÓDIGO 3.3] Rastreando las llamadas al sistema reales con `strace`
```bash
# Instalamos strace para interceptar las llamadas que hace el ejecutable al Kernel
!apt-get install -y strace > /dev/null 2>&1
!strace -e trace=write,getpid ./syscall_demo

In [ ]:
# Instalamos strace para interceptar las llamadas que hace el ejecutable al Kernel
!apt-get install -y strace > /dev/null 2>&1
!strace -e trace=write,getpid ./syscall_demo

getpid()                                = 6696
write(1, "--------------------------------"..., 51--------------------------------------------------
) = 51
write(1, "[MODO USUARIO] Ejecutando progra"..., 43[MODO USUARIO] Ejecutando programa en C...
) = 43
write(1, "[LLAMADA AL SISTEMA] PID del Pro"..., 50[LLAMADA AL SISTEMA] PID del Proceso Actual: 6696
) = 50
write(1, "[LLAMADA AL SISTEMA] PID del Pro"..., 63[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): 6693
) = 63
write(1, "--------------------------------"..., 51--------------------------------------------------
) = 51
+++ exited with 0 +++


## 6. Desafío Asíncrono / Entregable de la Unidad I

### Instrucciones:
1. Modifica el archivo `syscall_demo.c` de la Sección 5 creando un nuevo archivo `desafio_u1.c` mediante celdas mágicas `%%writefile`.
2. Agrega la llamada al sistema POSIX `getuid()` (para obtener el ID del usuario que ejecuta el proceso) y la función `sleep(5)` para pausar la ejecución durante 5 segundos.
3. Ejecuta el programa en segundo plano mediante Bash y utiliza el comando `ps aux` para capturar el proceso en la lista de procesos activos mientras duerme.
4. Vuelve a ejecutar `strace` sobre el nuevo binario para identificar las syscalls `nanosleep` y `getuid`.

---

In [ ]:
%%writefile desafio_u1.c
/*
 * Desafío de la Unidad 1 - Llamadas al Sistema POSIX extendidas
 * Este programa obtiene ID de proceso, ID de usuario y pausa la ejecución.
 */

#include <stdio.h>
#include <unistd.h>
#include <sys/types.h>

int main() {
    pid_t process_id = getpid();
    pid_t parent_id = getppid();

    // Nueva llamada al sistema: obtener ID de usuario
    uid_t user_id = getuid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C (Desafio U1)...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("[LLAMADA AL SISTEMA] UID del Usuario Actual: %d\n", user_id);
    printf("--------------------------------------------------\n");

    printf("[MODO USUARIO] El proceso dormirá por 5 segundos...\n");
    // Nueva llamada al sistema: pausar la ejecución
    sleep(5);
    printf("[MODO USUARIO] El proceso ha despertado y terminará.\n");

    return 0;
}

Writing desafio_u1.c


### Compilación de `desafio_u1.c`

In [ ]:
!gcc -Wall desafio_u1.c -o desafio_u1

### Ejecución en segundo plano y verificación con `ps aux`

In [ ]:
!./desafio_u1 & sleep 1; ps aux | grep desafio_u1 | grep -v grep

--------------------------------------------------
[MODO USUARIO] Ejecutando programa en C (Desafio U1)...
[LLAMADA AL SISTEMA] PID del Proceso Actual: 10247
[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): 10246
[LLAMADA AL SISTEMA] UID del Usuario Actual: 0
--------------------------------------------------
[MODO USUARIO] El proceso dormirá por 5 segundos...
root       10247  0.0  0.0   2776  1536 ?        S    05:30   0:00 ./desafio_u1
[MODO USUARIO] El proceso ha despertado y terminará.


### Rastreando las llamadas al sistema con `strace` para `desafio_u1`

In [ ]:
# Usamos strace para interceptar las llamadas relevantes al Kernel
!strace -e trace=nanosleep,getuid,getpid,write ./desafio_u1

getpid()                                = 11499
getuid()                                = 0
write(1, "--------------------------------"..., 51--------------------------------------------------
) = 51
write(1, "[MODO USUARIO] Ejecutando progra"..., 56[MODO USUARIO] Ejecutando programa en C (Desafio U1)...
) = 56
write(1, "[LLAMADA AL SISTEMA] PID del Pro"..., 51[LLAMADA AL SISTEMA] PID del Proceso Actual: 11499
) = 51
write(1, "[LLAMADA AL SISTEMA] PID del Pro"..., 64[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): 11496
) = 64
write(1, "[LLAMADA AL SISTEMA] UID del Usu"..., 47[LLAMADA AL SISTEMA] UID del Usuario Actual: 0
) = 47
write(1, "--------------------------------"..., 51--------------------------------------------------
) = 51
write(1, "[MODO USUARIO] El proceso dormir"..., 53[MODO USUARIO] El proceso dormirá por 5 segundos...
) = 53
write(1, "[MODO USUARIO] El proceso ha des"..., 54[MODO USUARIO] El proceso ha despertado y terminará.
) = 54
+++ exited with 0 +++


### Vuelve a ejecutar strace sobre el nuevo binario para identificar las syscalls `nanosleep` y `getuid`.

In [ ]:
!strace -e trace=getuid,clock_nanosleep,getpid ./desafio_u1

getpid()                                = 11907
getuid()                                = 0
--------------------------------------------------
[MODO USUARIO] Ejecutando programa en C (Desafio U1)...
[LLAMADA AL SISTEMA] PID del Proceso Actual: 11907
[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): 11904
[LLAMADA AL SISTEMA] UID del Usuario Actual: 0
--------------------------------------------------
[MODO USUARIO] El proceso dormirá por 5 segundos...
clock_nanosleep(CLOCK_REALTIME, 0, {tv_sec=5, tv_nsec=0}, 0x7ffd7cc2cda0) = 0
[MODO USUARIO] El proceso ha despertado y terminará.
+++ exited with 0 +++


### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas incluyendo las respuesta a las preguntas (`Archivo` -> `Guardar una copia en Drive`).
2. Sube los cambios a tu repositorio de GitHub de la materia bajo el nombre `JUAN PEREZ Practica1_Conceptos_SO.ipynb` usando la integración directa (`Archivo` -> `Guardar una copia en GitHub`).
3. Adjunta en tu entregable de plataforma Moodle, el enlace de GitHub.